# Comparer les branches de detection

Ce notebook lance les memes cas de test centralises contre les branches activees : `generic`, `spacy`, `gliner`, `regex`.

Toute la configuration (interrupteurs de branches, chemins des modeles, labels GLiNER, ...) est centralisee dans la premiere cellule de code : modifiez-la a un seul endroit, les cellules suivantes la reutilisent. Executez les cellules dans l'ordre.


In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd

def find_project_root(start_path: Path) -> Path:
    for candidate in (start_path, *start_path.parents):
        if (candidate / "src" / "compliance_nlp").exists() and (candidate / "configs").exists():
            return candidate
    raise FileNotFoundError("Impossible de trouver la racine du projet depuis le dossier courant.")


ROOT = find_project_root(Path.cwd().resolve())

SRC_DIR = ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

ENABLE_GENERIC_BRANCH = False
ENABLE_SPACY_BRANCH = False
ENABLE_GLINER_BRANCH = True
ENABLE_REGEX_BRANCH = False

MODEL_CACHE_DIR = r"D:\Workspaces\ModelCache"
MODEL_STORE_DIR = r"D:\Workspaces\ModelStore"
SPACY_MODEL = rf"{MODEL_STORE_DIR}\fr_core_news_md"
SPACY_SYNONYMS_PATH = ROOT / "configs" / "spacy_synonyms.csv"
JEUXDEMOTS_STORE_PATH = Path(MODEL_STORE_DIR) / "jeuxdemots"
GLINER_MODEL = rf"{MODEL_STORE_DIR}\gliner_multi-v2.1"
GLINER_SOURCE_MODEL = "urchade/gliner_multi-v2.1"
GLINER_THRESHOLD = 0.50
GLINER_LOCAL_FILES_ONLY = True
PROCESSING_CHUNK_SIZE = 512
SPACY_BATCH_SIZE = 256
GLINER_BATCH_SIZE = 16
# Tableau unique des labels Article 9 et de leurs distracteurs.
# Les formulations generiques trop ambigues ont ete remplacees par des types precis.
ARTICLE9_LABEL_TABLE = [
    {
        "groupe": "sante_clinique",
        "labels_sensibles": [
            "maladie diagnostiquee",
            "pathologie medicale",
            "symptome medical",
            "traitement medical",
            "antecedent medical",
            "resultat d examen medical",
        ],
        "distracteurs": [
            "civilite", "nom de personne", "numero de telephone",
            "nom de produit", "marque commerciale", "sigle metier",
            "information fiscale", "information patrimoniale",
        ],
    },
    {
        "groupe": "handicap_neurodeveloppement",
        "labels_sensibles": [
            "handicap", "situation de handicap", "trouble du spectre autistique",
            "trouble du neurodeveloppement", "perte d autonomie", "incapacite medicale",
        ],
        "distracteurs": [
            "produit d assurance", "produit financier", "nom de produit",
            "contrat", "personne a charge", "situation familiale", "sigle metier",
        ],
    },
    {
        "groupe": "opinion_politique",
        "labels_sensibles": [
            "opinion politique", "ideologie politique", "appartenance a un parti politique",
            "engagement politique", "militantisme politique",
        ],
        "distracteurs": [
            "direction spatiale", "position sur une piste", "orientation geographique",
            "nom de personne", "organisation",
            "information politique generale sans opinion personnelle",
        ],
    },
    {
        "groupe": "religion",
        "labels_sensibles": [
            "conviction religieuse", "croyance religieuse", "appartenance religieuse",
            "appartenance a une communaute religieuse", "pratique religieuse",
        ],
        "distracteurs": [
            "nom de personne", "sigle commercial", "sigle metier", "produit financier",
            "produit de retraite", "organisation", "lieu", "situation familiale",
        ],
    },
    {
        "groupe": "conviction_philosophique",
        "labels_sensibles": ["conviction philosophique", "croyance philosophique"],
        "distracteurs": [
            "opinion generale", "preference personnelle", "theme philosophique general",
            "nom de personne", "organisation",
        ],
    },
    {
        "groupe": "appartenance_syndicale",
        "labels_sensibles": [
            "appartenance syndicale", "adhesion a un syndicat", "mandat syndical",
            "engagement syndical", "activite syndicale", "responsabilite syndicale",
        ],
        "distracteurs": [
            "nom d organisation", "organisation professionnelle", "entreprise",
            "contrat", "produit financier", "sigle metier",
        ],
    },
    {
        "groupe": "orientation_sexuelle",
        "labels_sensibles": [
            "orientation sexuelle", "orientation homosexuelle", "orientation bisexuelle",
        ],
        "distracteurs": [
            "situation familiale", "situation matrimoniale", "conjoint ou conjointe",
            "civilite", "nom de personne",
        ],
    },
    {
        "groupe": "vie_sexuelle",
        "labels_sensibles": ["vie sexuelle", "information sur la vie sexuelle"],
        "distracteurs": [
            "situation familiale", "situation matrimoniale", "relation familiale",
            "conjoint ou conjointe", "nom de personne",
        ],
    },
    {
        "groupe": "origine_ethnique_raciale",
        "labels_sensibles": [
            "origine ethnique", "origine raciale", "appartenance ethnique", "groupe ethnique",
        ],
        "distracteurs": [
            "nationalite", "lieu de naissance", "adresse", "langue parlee",
            "nom de personne", "lieu geographique",
        ],
    },
    {
        "groupe": "donnee_genetique",
        "labels_sensibles": [
            "donnee genetique", "information genetique", "test genetique", "profil ADN",
        ],
        "distracteurs": [
            "analyse de laboratoire non genetique", "code administratif", "identifiant client",
            "numero de contrat", "nom de produit", "sigle metier",
        ],
    },
    {
        "groupe": "biometrie_identification",
        "labels_sensibles": [
            "donnee biometrique pour identification unique",
            "empreinte digitale pour identification",
            "reconnaissance faciale pour identification",
            "iris pour identification",
        ],
        "distracteurs": [
            "photographie ordinaire", "signature", "identifiant client",
            "numero de telephone", "numero de contrat", "code administratif",
        ],
    },
]

# Dictionnaires derives du tableau, au format attendu par le pipeline.
GLINER_LABEL_GROUPS = {
    row["groupe"]: list(dict.fromkeys(row["labels_sensibles"]))
    for row in ARTICLE9_LABEL_TABLE
}
GLINER_DISTRACTOR_GROUPS = {
    row["groupe"]: list(dict.fromkeys(row["distracteurs"]))
    for row in ARTICLE9_LABEL_TABLE
}

GLINER_PREDICTION_LABEL_GROUPS = {
    group_name: list(dict.fromkeys([*sensitive_labels, *GLINER_DISTRACTOR_GROUPS.get(group_name, [])]))
    for group_name, sensitive_labels in GLINER_LABEL_GROUPS.items()
}
GLINER_LABELS = [label for labels in GLINER_LABEL_GROUPS.values() for label in labels]
GLINER_DISTRACTOR_LABELS = {
    label.casefold()
    for labels in GLINER_DISTRACTOR_GROUPS.values()
    for label in labels
}
ARTICLE9_LABELS_DF = pd.DataFrame(ARTICLE9_LABEL_TABLE).rename(
    columns={"groupe": "groupe_article9"}
)
ARTICLE9_LABELS_DF["nb_labels_sensibles"] = ARTICLE9_LABELS_DF["labels_sensibles"].map(len)
ARTICLE9_LABELS_DF["nb_distracteurs"] = ARTICLE9_LABELS_DF["distracteurs"].map(len)
ARTICLE9_LABELS_DF = ARTICLE9_LABELS_DF[[
    "groupe_article9", "nb_labels_sensibles", "labels_sensibles",
    "nb_distracteurs", "distracteurs",
]]

pd.set_option("display.max_colwidth", 180)

os.environ["HF_HOME"] = MODEL_CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = rf"{MODEL_CACHE_DIR}\transformers"
os.environ["HUGGINGFACE_HUB_CACHE"] = rf"{MODEL_CACHE_DIR}\hub"
os.environ["COMPLIANCE_NLP_MODEL_CACHE"] = MODEL_CACHE_DIR
os.environ["COMPLIANCE_NLP_MODEL_STORE"] = MODEL_STORE_DIR

enabled_branches = tuple(
    branch
    for branch, enabled in {
        "generic": ENABLE_GENERIC_BRANCH,
        "spacy": ENABLE_SPACY_BRANCH,
        "gliner": ENABLE_GLINER_BRANCH,
        "regex": ENABLE_REGEX_BRANCH,
    }.items()
    if enabled
)

params = {
    "enabled_branches": enabled_branches,
    "model_cache_dir": MODEL_CACHE_DIR,
    "model_store_dir": MODEL_STORE_DIR,
    "spacy_model": SPACY_MODEL,
    "spacy_synonyms_path": str(SPACY_SYNONYMS_PATH),
    "gliner_model": GLINER_MODEL,
    "gliner_threshold": GLINER_THRESHOLD,
    "gliner_local_files_only": GLINER_LOCAL_FILES_ONLY,
    "processing_chunk_size": PROCESSING_CHUNK_SIZE,
    "spacy_batch_size": SPACY_BATCH_SIZE,
    "gliner_batch_size": GLINER_BATCH_SIZE,
    "gliner_label_groups": GLINER_LABEL_GROUPS,
    "gliner_distractor_groups": GLINER_DISTRACTOR_GROUPS,
    "gliner_prediction_label_groups": GLINER_PREDICTION_LABEL_GROUPS,
    "gliner_labels": GLINER_LABELS,
}
ARTICLE9_LABELS_DF


In [ ]:
import sys

# Toutes les variables de configuration (ROOT, chemins modeles, branches, labels GLiNER, ...)
# sont definies une seule fois dans la cellule precedente et reutilisees ici.
loaded_package = sys.modules.get("compliance_nlp")
if loaded_package is not None and not hasattr(loaded_package, "refresh_spacy_synonyms_from_forbidden_words"):
    for module_name in list(sys.modules):
        if module_name == "compliance_nlp" or module_name.startswith("compliance_nlp."):
            del sys.modules[module_name]

from compliance_nlp import (
    load_generic_detection_rules,
    load_whitelist_terms,
    refresh_spacy_synonyms_from_forbidden_words,
)
from compliance_nlp.pipeline import analyze_text, analyze_texts

spacy_synonyms_count = refresh_spacy_synonyms_from_forbidden_words(
    forbidden_words_path=ROOT / "configs" / "Mots_interdits.csv",
    synonyms_path=SPACY_SYNONYMS_PATH,
    jeuxdemots_store_path=JEUXDEMOTS_STORE_PATH,
    enrich_with_jeuxdemots=True,
    allow_network=False,
    max_synonyms=5,
    min_weight=25,
)


generic_rules = load_generic_detection_rules(ROOT / "configs" / "Mots_interdits.csv")
whitelist_terms = load_whitelist_terms(ROOT / "configs" / "article9_whitelist.csv")

referentials_df = pd.DataFrame([
    {"referentiel": "generic_rules", "count": len(generic_rules)},
    {"referentiel": "whitelist_terms", "count": len(whitelist_terms)},
    {"referentiel": "spacy_synonyms", "count": spacy_synonyms_count},
    {"referentiel": "gliner_label_groups", "count": len(GLINER_LABEL_GROUPS)},
    {"referentiel": "gliner_labels", "count": len(GLINER_LABELS)},
    {"referentiel": "gliner_distractor_labels", "count": len(GLINER_DISTRACTOR_LABELS)},
    {"referentiel": "gliner_prediction_labels", "count": sum(len(labels) for labels in GLINER_PREDICTION_LABEL_GROUPS.values())},
])
referentials_df


In [ ]:
TEST_CASES_PATH = ROOT / "configs" / "test_cases_new.csv"
TEST_CASE_COLUMNS = ["case_id", "text", "control_family", "comment"]

# sep=None + engine="python" detecte automatiquement le separateur (Excel bascule parfois , <-> ;)
test_cases = pd.read_csv(
    TEST_CASES_PATH,
    keep_default_na=False,
    sep=None,
    engine="python",
    encoding="utf-8-sig",
)
missing_columns = set(TEST_CASE_COLUMNS) - set(test_cases.columns)
if missing_columns:
    raise ValueError(f"Colonnes manquantes dans {TEST_CASES_PATH}: {sorted(missing_columns)}")

test_cases = test_cases[TEST_CASE_COLUMNS].copy()
test_cases = test_cases[["case_id", "text", "control_family", "comment"]]
#test_cases


In [ ]:
def finding_key(finding):
    return finding.rule_id or finding.code


def is_gliner_distractor_finding(finding):
    return (
        finding.detection_engine == "gliner"
        and (finding.title or "").strip().casefold() in GLINER_DISTRACTOR_LABELS
    )


def is_gliner_distractor_payload(finding):
    return (
        finding.get("detection_engine") == "gliner"
        and str(finding.get("title") or "").strip().casefold() in GLINER_DISTRACTOR_LABELS
    )


def count_findings_by_engine(active_findings, ignored_findings=()):
    counts = {}
    for finding in active_findings:
        engine = finding.detection_engine or "unknown"
        counts[engine] = counts.get(engine, 0) + 1
    for ignored in ignored_findings:
        payload = ignored.get("finding", {})
        engine = payload.get("detection_engine") or "unknown"
        counts[engine] = counts.get(engine, 0) + 1
    return counts


def filter_gliner_distractors(analysis):
    original_active = list(analysis.findings)
    original_ignored = list(analysis.metadata.get("whitelist_ignored_findings", []))
    retained_active = [finding for finding in original_active if not is_gliner_distractor_finding(finding)]
    retained_ignored = [
        ignored
        for ignored in original_ignored
        if not is_gliner_distractor_payload(ignored.get("finding", {}))
    ]

    analysis.findings = retained_active
    metadata = analysis.metadata
    metadata["finding_count"] = len(retained_active)
    metadata["has_findings"] = bool(retained_active)
    metadata["whitelist_ignored_findings"] = retained_ignored
    metadata["whitelist_ignored_count"] = len(retained_ignored)
    metadata["raw_finding_count"] = len(retained_active) + len(retained_ignored)
    metadata["raw_finding_count_by_engine"] = count_findings_by_engine(retained_active, retained_ignored)
    metadata["whitelist_ignored_count_by_engine"] = count_findings_by_engine([], retained_ignored)
    metadata["gliner_filtered_distractor_count"] = (
        len(original_active) + len(original_ignored) - len(retained_active) - len(retained_ignored)
    )
    metadata["gliner_labels"] = list(GLINER_LABELS)
    metadata["gliner_label_groups"] = GLINER_LABEL_GROUPS
    metadata["gliner_distractor_label_groups"] = GLINER_DISTRACTOR_GROUPS
    metadata["gliner_prediction_label_groups"] = GLINER_PREDICTION_LABEL_GROUPS

    for engine in ("generic", "spacy", "gliner", "regex"):
        engine_findings = [finding for finding in retained_active if finding.detection_engine == engine]
        scores = [
            getattr(finding, f"{engine}_score")
            for finding in engine_findings
            if getattr(finding, f"{engine}_score") is not None
        ]
        metadata[f"{engine}_finding_count"] = len(engine_findings)
        metadata[f"{engine}_max_score"] = max(scores, default=None)

    return analysis


def run_detector(text: str, case_id: str):
    analysis = analyze_text(
        document_name=case_id,
        source_path="notebook",
        extracted_text=text,
        generic_rules=generic_rules,
        whitelist_terms=whitelist_terms,
        enabled_branches=enabled_branches,
        spacy_model=SPACY_MODEL,
        spacy_synonyms_path=str(SPACY_SYNONYMS_PATH),
        gliner_model=GLINER_MODEL,
        gliner_cache_dir=MODEL_CACHE_DIR,
        gliner_source_model=GLINER_SOURCE_MODEL,
        gliner_labels=GLINER_PREDICTION_LABEL_GROUPS,
        gliner_threshold=GLINER_THRESHOLD,
        gliner_local_files_only=GLINER_LOCAL_FILES_ONLY,
    )
    return filter_gliner_distractors(analysis)


def run_detector_batch(cases):
    documents = [
        (str(case["case_id"]), "notebook", str(case["text"]))
        for case in cases
    ]
    analyses = analyze_texts(
        documents,
        generic_rules=generic_rules,
        whitelist_terms=whitelist_terms,
        enabled_branches=enabled_branches,
        spacy_model=SPACY_MODEL,
        spacy_synonyms_path=str(SPACY_SYNONYMS_PATH),
        spacy_batch_size=SPACY_BATCH_SIZE,
        gliner_model=GLINER_MODEL,
        gliner_cache_dir=MODEL_CACHE_DIR,
        gliner_source_model=GLINER_SOURCE_MODEL,
        gliner_labels=GLINER_PREDICTION_LABEL_GROUPS,
        gliner_threshold=GLINER_THRESHOLD,
        gliner_local_files_only=GLINER_LOCAL_FILES_ONLY,
        gliner_batch_size=GLINER_BATCH_SIZE,
        deduplicate_texts=True,
    )
    return [filter_gliner_distractors(analysis) for analysis in analyses]


def empty_row(case, engine):
    return {
        "case_id": case["case_id"],
        "control_family": case["control_family"],
        "comment": case["comment"],
        "detection_status": "none",
        "decision": "aucune detection",
        "ignored_by_whitelist": False,
        "whitelist_expression": None,
        "whitelist_reason": None,
        "detection_engine": engine,
        "predicted_key": None,
        "code": None,
        "rule_id": None,
        "rule_scope": None,
        "regulatory_family": None,
        "section": None,
        "category": None,
        "matched_term": None,
        "detection_type": None,
        "score": None,
        "branch_score": None,
        "generic_score": None,
        "spacy_score": None,
        "gliner_score": None,
        "regex_score": None,
        "severity": None,
        "evidence": None,
    }


def finding_dict_to_row(case, finding, detection_status="active", whitelist_expression=None, whitelist_reason=None):
    return {
        "case_id": case["case_id"],
        "control_family": case["control_family"],
        "comment": case["comment"],
        "detection_status": detection_status,
        "decision": "detecte puis ignore par whitelist" if detection_status == "ignored_by_whitelist" else "alerte active",
        "ignored_by_whitelist": detection_status == "ignored_by_whitelist",
        "whitelist_expression": whitelist_expression,
        "whitelist_reason": whitelist_reason,
        "detection_engine": finding.get("detection_engine"),
        "predicted_key": finding.get("rule_id") or finding.get("code"),
        "code": finding.get("code"),
        "rule_id": finding.get("rule_id"),
        "rule_scope": finding.get("rule_scope"),
        "regulatory_family": finding.get("regulatory_family"),
        "section": finding.get("section"),
        "category": finding.get("category"),
        "matched_term": finding.get("matched_term"),
        "detection_type": finding.get("detection_type"),
        "score": finding.get("score"),
        "branch_score": finding.get("branch_score"),
        "generic_score": finding.get("generic_score"),
        "spacy_score": finding.get("spacy_score"),
        "gliner_score": finding.get("gliner_score"),
        "regex_score": finding.get("regex_score"),
        "severity": finding.get("severity"),
        "evidence": finding.get("evidence"),
    }


def findings_to_rows(case, analysis):
    rows = []
    for finding in analysis.findings:
        rows.append(finding_dict_to_row(case, finding.to_dict()))
    for ignored in analysis.metadata.get("whitelist_ignored_findings", []):
        rows.append(
            finding_dict_to_row(
                case,
                ignored["finding"],
                detection_status="ignored_by_whitelist",
                whitelist_expression=ignored.get("whitelist_expression"),
                whitelist_reason=ignored.get("whitelist_reason"),
            )
        )
    if not rows:
        return [empty_row(case, "none")]
    return rows


In [ ]:
rows = []
metadata_rows = []

for chunk_start in range(0, len(test_cases), PROCESSING_CHUNK_SIZE):
    chunk = test_cases.iloc[chunk_start : chunk_start + PROCESSING_CHUNK_SIZE]
    cases = [case._asdict() for case in chunk.itertuples(index=False)]
    chunk_analyses = run_detector_batch(cases)

    for case, analysis in zip(cases, chunk_analyses):
        rows.extend(findings_to_rows(case, analysis))
        metadata_rows.append({
            "case_id": case["case_id"],
            "control_family": case["control_family"],
            "comment": case["comment"],
            "enabled_branches": " | ".join(analysis.metadata.get("enabled_branches", [])),
            "raw_finding_count": analysis.metadata.get("raw_finding_count"),
            "finding_count": analysis.metadata.get("finding_count"),
            "whitelist_ignored_count": analysis.metadata.get("whitelist_ignored_count"),
            "generic_finding_count": analysis.metadata.get("generic_finding_count"),
            "spacy_finding_count": analysis.metadata.get("spacy_finding_count"),
            "gliner_finding_count": analysis.metadata.get("gliner_finding_count"),
            "gliner_filtered_distractor_count": analysis.metadata.get("gliner_filtered_distractor_count"),
            "regex_finding_count": analysis.metadata.get("regex_finding_count"),
            "generic_max_score": analysis.metadata.get("generic_max_score"),
            "spacy_max_score": analysis.metadata.get("spacy_max_score"),
            "gliner_max_score": analysis.metadata.get("gliner_max_score"),
            "regex_max_score": analysis.metadata.get("regex_max_score"),
            "raw_finding_count_by_engine": analysis.metadata.get("raw_finding_count_by_engine"),
            "whitelist_ignored_count_by_engine": analysis.metadata.get("whitelist_ignored_count_by_engine"),
            "branch_errors": analysis.metadata.get("branch_errors"),
        })

results_df = pd.DataFrame(rows)
case_summary_df = pd.DataFrame(metadata_rows)
#case_summary_df


In [ ]:
alerts_df = results_df[results_df["predicted_key"].notna()].copy()
alerts_df.sort_values(["case_id", "detection_status", "detection_engine", "score"], ascending=[True, True, True, False])


In [ ]:
whitelist_ignored_df = alerts_df[alerts_df["ignored_by_whitelist"]].copy()
whitelist_ignored_df[[
    "case_id",
    "decision",
    "detection_engine",
    "predicted_key",
    "matched_term",
    "score",
    "whitelist_expression",
    "whitelist_reason",
    "evidence",
]].sort_values(["case_id", "detection_engine", "score"], ascending=[True, True, False])


In [ ]:
def join_unique_strings(values):
    cleaned = {str(value) for value in values if isinstance(value, str) and value}
    return " | ".join(sorted(cleaned))


comparison_df = (
    alerts_df.groupby(["case_id", "control_family", "comment", "detection_status", "decision", "detection_engine"], dropna=False)
    .agg(
        detection_count=("predicted_key", "size"),
        max_score=("score", "max"),
        predicted_keys=("predicted_key", join_unique_strings),
        matched_terms=("matched_term", join_unique_strings),
        whitelist_expressions=("whitelist_expression", join_unique_strings),
    )
    .reset_index()
)
#comparison_df


In [ ]:
count_pivot_df = comparison_df.pivot_table(
    index=["case_id", "control_family", "comment", "detection_status", "decision"],
    columns="detection_engine",
    values="detection_count",
    aggfunc="sum",
    fill_value=0,
).reset_index()
#count_pivot_df


In [ ]:
score_columns = ["generic_score", "spacy_score", "gliner_score", "regex_score"]
score_view_df = alerts_df[[
    "case_id",
    "detection_status",
    "decision",
    "ignored_by_whitelist",
    "detection_engine",
    "predicted_key",
    "matched_term",
    "detection_type",
    *score_columns,
    "whitelist_expression",
    "whitelist_reason",
    "evidence",
]].sort_values(["case_id", "detection_status", "detection_engine", "predicted_key"], na_position="last")
#score_view_df


In [ ]:
branch_errors_df = case_summary_df[["case_id", "branch_errors"]].copy()
branch_errors_df = branch_errors_df[branch_errors_df["branch_errors"].map(bool)]
branch_errors_df


## Export des resultats

Cette cellule ecrit les vues principales du notebook dans `outputs/comparaison_branches` pour permettre une reprise dans Excel ou dans le notebook d'export multi-onglets.


In [ ]:
export_dir = ROOT / "outputs" / "comparaison_branches"
export_dir.mkdir(parents=True, exist_ok=True)

exports = {
    "test_cases": test_cases,
    "referentials": referentials_df,
    "case_summary": case_summary_df,
    "results_all": results_df,
    "alerts": alerts_df,
    "whitelist_ignored": whitelist_ignored_df,
    "comparison": comparison_df,
    "count_pivot": count_pivot_df,
    "score_view": score_view_df,
    "branch_errors": branch_errors_df,
}

export_paths = []
for name, dataframe in exports.items():
    path = export_dir / f"{name}.csv"
    dataframe.to_csv(path, index=False, encoding="utf-8-sig")
    export_paths.append({
        "name": name,
        "rows": len(dataframe),
        "columns": len(dataframe.columns),
        "path": str(path),
    })

export_manifest_df = pd.DataFrame(export_paths)
export_manifest_path = export_dir / "manifest.csv"
export_manifest_df.to_csv(export_manifest_path, index=False, encoding="utf-8-sig")
export_manifest_df
